In [2]:
import pandas as pd

df = pd.read_csv('../data/archive/Disease_symptom_and_patient_profile_dataset.csv')
print(df.shape)      # how many rows and columns
print(df.columns)    # what the column names are
print(df.head())     # first 5 rows

(349, 10)
Index(['Disease', 'Fever', 'Cough', 'Fatigue', 'Difficulty Breathing', 'Age',
       'Gender', 'Blood Pressure', 'Cholesterol Level', 'Outcome Variable'],
      dtype='str')
       Disease Fever Cough Fatigue Difficulty Breathing  Age  Gender  \
0    Influenza   Yes    No     Yes                  Yes   19  Female   
1  Common Cold    No   Yes     Yes                   No   25  Female   
2       Eczema    No   Yes     Yes                   No   25  Female   
3       Asthma   Yes   Yes      No                  Yes   25    Male   
4       Asthma   Yes   Yes      No                  Yes   25    Male   

  Blood Pressure Cholesterol Level Outcome Variable  
0            Low            Normal         Positive  
1         Normal            Normal         Negative  
2         Normal            Normal         Negative  
3         Normal            Normal         Positive  
4         Normal            Normal         Positive  


In [21]:
import sqlite3

# This creates symptoms.db inside your data folder
# If it already exists it just connects to it
conn = sqlite3.connect('../data/symptoms.db')
cursor = conn.cursor()

# Create the diseases table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS diseases (
        disease_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE NOT NULL
    )
""")

# Create the symptoms table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS symptoms (
        symptom_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL
    )
""")

# Create the linking table that connects diseases to symptoms
cursor.execute("""
    CREATE TABLE IF NOT EXISTS disease_symptoms (
        disease_id INTEGER,
        symptom_id INTEGER,
        FOREIGN KEY (disease_id) REFERENCES diseases(disease_id),
        FOREIGN KEY (symptom_id) REFERENCES symptoms(symptom_id)
    )
""")

conn.commit()
print("Tables created successfully")

Tables created successfully


In [28]:
# First get all unique diseases and insert them, and only keep unique copies each run

unique_diseases = df['Disease'].unique()

print(unique_diseases)

for disease in unique_diseases:
    cursor.execute("""
        INSERT OR IGNORE INTO diseases (name) VALUES (?)
    """, (disease,))

for disease in unique_diseases:
    cursor.execute("""
        DELETE FROM diseases
            WHERE disease_id NOT IN (
                SELECT MIN(disease_id)
                FROM diseases
                GROUP BY name
        )
    """)

conn.commit()
print(f"Inserted {len(unique_diseases)} diseases")

<ArrowStringArray>
[        'Influenza',       'Common Cold',            'Eczema',
            'Asthma',   'Hyperthyroidism', 'Allergic Rhinitis',
 'Anxiety Disorders',          'Diabetes',   'Gastroenteritis',
      'Pancreatitis',
 ...
          'Dementia',    'Diverticulitis',             'Mumps',
     'Cholecystitis',   'Prostate Cancer',     'Schizophrenia',
              'Gout', 'Testicular Cancer',       'Tonsillitis',
 'Williams Syndrome']
Length: 116, dtype: str
Inserted 116 diseases


In [29]:
# Then get all unique symptoms and insert them
# Your CSV might have columns like Symptom1, Symptom2, Symptom3

# Still have to deal w the types/potential outcomes of each symptom (yes/no, age, etc.)

symptom_columns = ['Fever', 'Cough', 'Fatigue', 'Difficulty Breathing', 'Age',
                   'Gender', 'Blood Pressure', 'Cholesterol Level', 'Outcome Variable']

for symptom in symptom_columns:
    cursor.execute("""
        INSERT INTO symptoms (name) VALUES (?)
    """, (symptom,))

for symptom in symptom_columns:
    cursor.execute("""
        DELETE FROM symptoms
            WHERE symptom_id NOT IN (
                SELECT MIN(symptom_id)
                FROM symptoms
                GROUP BY name
        )
    """)

conn.commit()
print(f"Inserted {len(symptom_columns)} symptoms")

Inserted 9 symptoms


In [30]:
# Then populate the linking table

'''

for _, row in df.iterrows():
    # Get the disease_id for this row's disease
    cursor.execute("SELECT disease_id FROM diseases WHERE name = ?", (row['Disease'],))
    disease_id = cursor.fetchone()[0]
    
    # For each symptom in this row, link it to the disease
    for col in symptom_columns:
        if pd.notna(row[col]):
            cursor.execute("SELECT symptom_id FROM symptoms WHERE name = ?", (row[col],))
            symptom_id = cursor.fetchone()[0]
            
            cursor.execute("""
                INSERT INTO disease_symptoms (disease_id, symptom_id)
                VALUES (?, ?)
            """, (disease_id, symptom_id))

conn.commit()
print("Linking table populated")

'''

'\n\nfor _, row in df.iterrows():\n    # Get the disease_id for this row\'s disease\n    cursor.execute("SELECT disease_id FROM diseases WHERE name = ?", (row[\'Disease\'],))\n    disease_id = cursor.fetchone()[0]\n\n    # For each symptom in this row, link it to the disease\n    for col in symptom_columns:\n        if pd.notna(row[col]):\n            cursor.execute("SELECT symptom_id FROM symptoms WHERE name = ?", (row[col],))\n            symptom_id = cursor.fetchone()[0]\n\n            cursor.execute("""\n                INSERT INTO disease_symptoms (disease_id, symptom_id)\n                VALUES (?, ?)\n            """, (disease_id, symptom_id))\n\nconn.commit()\nprint("Linking table populated")\n\n'